#importing the dataset

In [90]:
import numpy as np
import pandas as pd


#Load the Dataset

In [91]:
df=pd.read_csv("/content/Heart_Disease_Prediction.csv")
df.head()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,Presence
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,Absence
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,Presence
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,Absence
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,Absence


In [92]:
df.isnull().sum()

,0
Age,0
Sex,0
Chest pain type,0
BP,0
Cholesterol,0
FBS over 120,0
EKG results,0
Max HR,0
Exercise angina,0
ST depression,0


In [93]:
df["Heart Disease"]=df["Heart Disease"].astype(str)
df["Heart Disease"] = df["Heart Disease"].str.strip().str.lower()
df["Heart Disease"] = df["Heart Disease"].replace({
    "presence": 1,
    "absence": 0
})
print("Before filtering:", df.shape)
df=df[df["Heart Disease"].isin([0,1])]
print("After filtering:", df.shape)
print(df["Heart Disease"].value_counts())

Before filtering: (270, 14)
After filtering: (270, 14)
Heart Disease
0    150
1    120
Name: count, dtype: int64


/tmp/ipykernel_1230/2787067367.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Heart Disease"] = df["Heart Disease"].replace({


In [94]:
df.fillna(df.mean(numeric_only=True), inplace=True)

In [99]:
X=df.drop("Heart Disease", axis=1)
y=df["Heart Disease"]

#Spliting the data in traing and testing

In [100]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)

#Standard Scaling

In [101]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)

#Bagging

#1.Random Forest

In [102]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier()
rf.fit(x_train,y_train)
rf_pred=rf.predict(x_test)

#2.Gradient Boosting

In [103]:
from sklearn.ensemble import GradientBoostingClassifier
gb=GradientBoostingClassifier()
gb.fit(x_train,y_train)
gb_pred=gb.predict(x_test)


#Logistic Regression

In [104]:
from sklearn.linear_model import LogisticRegression
lr=LogisticRegression(max_iter=1000)
lr.fit(x_train,y_train)
lr_pred=lr.predict(x_test)

#Voting Classifier

In [105]:
from sklearn.ensemble import VotingClassifier
ensemble=VotingClassifier(estimators=[('lr', lr),('rf', rf),('gb', gb)],voting='hard')
ensemble.fit(x_train,y_train)
ensemble_pred=ensemble.predict(x_test)

#Metrices

In [106]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print(" LR Accuracy:", accuracy_score(y_test,lr_pred))
print("GB Accuracy:", accuracy_score(y_test,gb_pred))
print("rf Accuracy:", accuracy_score(y_test,rf_pred))
print("stacking Accuracy:", accuracy_score(y_test, ensemble_pred))

 LR Accuracy: 0.9074074074074074
GB Accuracy: 0.7592592592592593
rf Accuracy: 0.8703703703703703
stacking Accuracy: 0.8703703703703703


In [107]:
import joblib
joblib.dump(ensemble, "voting_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Voting model saved successfully!")

Voting model saved successfully!


In [108]:
import joblib
model = joblib.load("voting_model.pkl")
print("Model loaded!")

Model loaded!


In [109]:
sample=[[63,1,3,145,233,1,0,150,0,2.3,0,0,1]]
sample_scaled = scaler.transform(sample)
prediction = ensemble.predict(sample_scaled)
print("Prediction:", prediction[0])
if prediction[0] == 1:
  print("The person has heart disease")
else:
  print("The person has no heart disease")


Prediction: 0
The person has no heart disease


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [110]:
sample=pd.DataFrame([[65, 1, 4, 160, 300, 1, 2, 120, 1, 3.0, 2, 2, 3]],columns=X.columns)
sample_scaled=scaler.transform(sample)
prediction=ensemble.predict(sample_scaled)
print("Prediction:", prediction[0])
if prediction[0]==1:
  print("The person has heart disease")
else:
  print("The person has no heart disease")

Prediction: 1
The person has heart disease
